# Notebook 06 — Cross-Model SHAP Agreement (Krishna Metrics)

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection
**Author:** Md Anas Biswas, University of Portsmouth
**Stage:** 6 of 10

## What this notebook does

Applies the six metrics from Krishna et al. 2022 (TMLR) to quantify cross-model SHAP agreement across Random Forest, XGBoost, and DNN architectures. This is novelty C2 of the paper.

## The six Krishna metrics

All operate on per-sample top-k feature lists from two explainers.

| Metric | What it measures | Range |
|---|---|---|
| **Feature Agreement** | Fraction of top-k features that appear in both explainers' top-k | [0, 1] |
| **Rank Agreement** | Fraction of top-k features that appear in **same rank position** in both | [0, 1] |
| **Sign Agreement** | Fraction of top-k features with same sign of SHAP in both | [0, 1] |
| **Signed Rank Agreement** | Fraction of top-k features with same sign **and** same rank | [0, 1] |
| **Rank Correlation** | Spearman correlation between the two explainers' rankings of top-k features | [-1, 1] |
| **Pairwise Rank Agreement** | For any pair of top-k features, do both explainers agree on which is more important? | [0, 1] |

## Methodology

- **Sample basis:** the 2,433 RF subsample (same indices used in Notebook 04 for RF SHAP). The other models' SHAP is sliced to these same indices for apples-to-apples comparison.
- **k values:** 5, 10, 15 (robustness check)
- **Granularity:** aggregate (binary + 5-class) AND per-class (for 5-class)
- **Model pairs:** RF↔XGB, RF↔DNN, XGB↔DNN

## Output files

```
results/tables/
├── nslkdd_krishna_aggregate.csv        # 3 pairs × 6 metrics × 3 k-values × 2 targets
├── nslkdd_krishna_perclass.csv         # adds per-class rows for 5-class
└── nslkdd_krishna_summary.csv          # headline numbers for paper
results/figures/
├── nslkdd_krishna_heatmap_binary.png
├── nslkdd_krishna_heatmap_5class.png
└── nslkdd_krishna_k_robustness.png
```

---
## Session start

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from scipy.stats import spearmanr

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

# Paths
PROCESSED = Path(REPO) / 'data' / 'processed' / 'nsl_kdd'
CALIB_DIR = Path(REPO) / 'calibrators' / 'nsl_kdd'
SHAP_DIR = Path(REPO) / 'shap_values' / 'nsl_kdd'
FIG_DIR = Path(REPO) / 'results' / 'figures'
TABLES_DIR = Path(REPO) / 'results' / 'tables'
for d in [FIG_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Load labels
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')
idx_eval = np.load(CALIB_DIR / 'idx_eval.npy')
rf_subsample_idx = np.load(SHAP_DIR / 'rf_subsample_idx.npy')

# The eval-half labels first, then RF subsample
y_eval_b = y_test_b[idx_eval]
y_eval_5 = y_test_5[idx_eval]
y_sub_b = y_eval_b[rf_subsample_idx]
y_sub_5 = y_eval_5[rf_subsample_idx]

with open(PROCESSED / 'feature_names.json') as f:
    FEATURE_NAMES = json.load(f)
with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'RF subsample: {len(rf_subsample_idx):,} samples')
print(f'Binary classes in subsample: {np.bincount(y_sub_b)}')
print(f'5-class in subsample: {np.bincount(y_sub_5)}')

RF subsample: 2,433 samples
Binary classes in subsample: [ 600 1833]
5-class in subsample: [600 600 600 600  33]


---
## Step 1 — Load and align SHAP arrays

RF was computed on the 2,433 subsample. Other models on the full 11,272. We slice the others to match RF's indices.

In [3]:
MODELS_BINARY = ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw']
MODELS_5CLASS = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote']
PAIRS = [('rf', 'xgb'), ('rf', 'dnn'), ('xgb', 'dnn')]

def load_and_align(name):
    '''Load SHAP and slice to RF subsample indices if needed.'''
    arr = np.load(SHAP_DIR / f'{name}_shap.npy')
    if name.startswith('rf'):
        return arr  # already 2,433 samples
    return arr[rf_subsample_idx]

SHAP_BINARY = {arch.split('_')[0]: load_and_align(arch) for arch in MODELS_BINARY}
SHAP_5CLASS = {arch.split('_')[0]: load_and_align(arch) for arch in MODELS_5CLASS}

print('Aligned SHAP shapes:')
print(f'\nBinary:')
for arch, sv in SHAP_BINARY.items():
    print(f'  {arch}: {sv.shape}')
print(f'\n5-class:')
for arch, sv in SHAP_5CLASS.items():
    print(f'  {arch}: {sv.shape}')

Aligned SHAP shapes:

Binary:
  rf: (2433, 122, 2)
  xgb: (2433, 122, 2)
  dnn: (2433, 122, 2)

5-class:
  rf: (2433, 122, 5)
  xgb: (2433, 122, 5)
  dnn: (2433, 122, 5)


---
## Step 2 — The six Krishna metrics

Each function takes two per-sample SHAP vectors (one per explainer) and a `k`, returns a scalar in [0,1] (or [-1,1] for rank correlation).

In [4]:
def feature_agreement(shap_a, shap_b, k):
    '''|top_k(A) ∩ top_k(B)| / k. Range [0, 1].'''
    top_a = set(np.argsort(-np.abs(shap_a))[:k])
    top_b = set(np.argsort(-np.abs(shap_b))[:k])
    return len(top_a & top_b) / k

def rank_agreement(shap_a, shap_b, k):
    '''Fraction of top-k features in same rank position in both.'''
    rank_a = np.argsort(-np.abs(shap_a))[:k]
    rank_b = np.argsort(-np.abs(shap_b))[:k]
    same_rank = sum(1 for i in range(k) if rank_a[i] == rank_b[i])
    return same_rank / k

def sign_agreement(shap_a, shap_b, k):
    '''Of top-k features common to both, what fraction have same sign?'''
    top_a = set(np.argsort(-np.abs(shap_a))[:k])
    top_b = set(np.argsort(-np.abs(shap_b))[:k])
    common = top_a & top_b
    if not common:
        return 0.0
    same_sign = sum(1 for f in common if np.sign(shap_a[f]) == np.sign(shap_b[f]))
    return same_sign / len(common)

def signed_rank_agreement(shap_a, shap_b, k):
    '''Fraction of top-k positions where same feature AND same sign.'''
    rank_a = np.argsort(-np.abs(shap_a))[:k]
    rank_b = np.argsort(-np.abs(shap_b))[:k]
    matches = sum(1 for i in range(k)
                  if rank_a[i] == rank_b[i] and np.sign(shap_a[rank_a[i]]) == np.sign(shap_b[rank_a[i]]))
    return matches / k

def rank_correlation(shap_a, shap_b, k):
    '''Spearman correlation between rankings of top-k features in A.
    (Standard practice: take A's top-k, get both A's and B's rank of those features, correlate.)'''
    top_a_idx = np.argsort(-np.abs(shap_a))[:k]
    # Ranks within these k features (1 = highest)
    ranks_a = np.argsort(np.argsort(-np.abs(shap_a)[top_a_idx])) + 1
    ranks_b = np.argsort(np.argsort(-np.abs(shap_b)[top_a_idx])) + 1
    if k < 2:
        return 0.0
    corr, _ = spearmanr(ranks_a, ranks_b)
    return float(corr) if not np.isnan(corr) else 0.0

def pairwise_rank_agreement(shap_a, shap_b, k):
    '''For each pair of features in top-k(A), do both explainers agree on which is more important?'''
    top_a = np.argsort(-np.abs(shap_a))[:k]
    abs_a = np.abs(shap_a)
    abs_b = np.abs(shap_b)
    n_pairs = 0
    agree = 0
    for i in range(k):
        for j in range(i+1, k):
            fi, fj = top_a[i], top_a[j]
            n_pairs += 1
            # A's view: i is more important than j (by construction since i < j in ranked order)
            # B's view: is abs_b[fi] >= abs_b[fj]?
            if abs_b[fi] >= abs_b[fj]:
                agree += 1
    return agree / n_pairs if n_pairs > 0 else 0.0

METRICS = {
    'feature_agreement': feature_agreement,
    'rank_agreement': rank_agreement,
    'sign_agreement': sign_agreement,
    'signed_rank_agreement': signed_rank_agreement,
    'rank_correlation': rank_correlation,
    'pairwise_rank_agreement': pairwise_rank_agreement,
}

# Quick sanity test on a small example
test_a = np.array([5, -3, 2, 1, -4, 0.5, -2, 1.5])
test_b = np.array([4, -3, 2, 1, -5, 0.5, -2, 1.5])  # almost the same
test_c = -test_a  # complete sign flip
print('Sanity check (k=4):')
for mname, fn in METRICS.items():
    print(f'  {mname:<25}  A vs A_similar={fn(test_a, test_b, 4):.3f}  A vs flipped={fn(test_a, test_c, 4):.3f}')

Sanity check (k=4):
  feature_agreement          A vs A_similar=1.000  A vs flipped=1.000
  rank_agreement             A vs A_similar=0.500  A vs flipped=1.000
  sign_agreement             A vs A_similar=1.000  A vs flipped=0.000
  signed_rank_agreement      A vs A_similar=0.500  A vs flipped=0.000
  rank_correlation           A vs A_similar=0.800  A vs flipped=1.000
  pairwise_rank_agreement    A vs A_similar=0.833  A vs flipped=1.000


---
## Step 3 — Compute metrics for binary models

For binary, SHAP shape is `(n, f, 2)`. We use the class-1 (Attack) attribution per sample.

In [5]:
K_VALUES = [5, 10, 15]

def get_binary_shap_vector(shap_arr, sample_i):
    '''For binary models, return per-sample SHAP vector for class 1 (Attack).'''
    if shap_arr.ndim == 3:
        return shap_arr[sample_i, :, 1]  # Attack class
    return shap_arr[sample_i]

def compute_pair_metrics_binary(arr_a, arr_b, k):
    '''Compute all 6 metrics averaged across samples for a model pair (binary).'''
    n = arr_a.shape[0]
    out = {m: 0.0 for m in METRICS}
    for i in range(n):
        sv_a = get_binary_shap_vector(arr_a, i)
        sv_b = get_binary_shap_vector(arr_b, i)
        for m, fn in METRICS.items():
            out[m] += fn(sv_a, sv_b, k)
    return {m: out[m] / n for m in METRICS}

print('Computing Krishna metrics — binary models...\n')
binary_results = []
for (arch_a, arch_b) in PAIRS:
    arr_a = SHAP_BINARY[arch_a]
    arr_b = SHAP_BINARY[arch_b]
    for k in K_VALUES:
        m = compute_pair_metrics_binary(arr_a, arr_b, k)
        row = {'Pair': f'{arch_a}-{arch_b}', 'k': k, 'Class': 'all', 'Target': 'binary'}
        row.update(m)
        binary_results.append(row)
        print(f'  {arch_a}-{arch_b} @ k={k}: ' + ', '.join(f'{mn[:8]}={v:.3f}' for mn, v in m.items()))

df_binary = pd.DataFrame(binary_results)

Computing Krishna metrics — binary models...

  rf-xgb @ k=5: feature_=0.575, rank_agr=0.267, sign_agr=0.999, signed_r=0.267, rank_cor=0.566, pairwise=0.736
  rf-xgb @ k=10: feature_=0.636, rank_agr=0.162, sign_agr=0.998, signed_r=0.162, rank_cor=0.470, pairwise=0.677
  rf-xgb @ k=15: feature_=0.662, rank_agr=0.124, sign_agr=0.972, signed_r=0.123, rank_cor=0.466, pairwise=0.669
  rf-dnn @ k=5: feature_=0.226, rank_agr=0.046, sign_agr=0.769, signed_r=0.044, rank_cor=-0.395, pairwise=0.350
  rf-dnn @ k=10: feature_=0.401, rank_agr=0.045, sign_agr=0.949, signed_r=0.043, rank_cor=-0.298, pairwise=0.394
  rf-dnn @ k=15: feature_=0.516, rank_agr=0.044, sign_agr=0.935, signed_r=0.042, rank_cor=-0.137, pairwise=0.456
  xgb-dnn @ k=5: feature_=0.292, rank_agr=0.073, sign_agr=0.901, signed_r=0.072, rank_cor=-0.295, pairwise=0.375
  xgb-dnn @ k=10: feature_=0.447, rank_agr=0.058, sign_agr=0.948, signed_r=0.057, rank_cor=-0.169, pairwise=0.440
  xgb-dnn @ k=15: feature_=0.511, rank_agr=0.044, sign

---
## Step 4 — Compute metrics for 5-class models (aggregate + per-class)

Aggregate: use sum of |SHAP| across classes as per-sample feature importance, then rank.

Per-class: use SHAP for each specific class. Subset samples to those of that true class.

In [ ]:
def get_5class_aggregate_vector(shap_arr, sample_i):
    '''Sum |SHAP| across classes for aggregate importance.'''
    return np.abs(shap_arr[sample_i]).sum(axis=-1)  # (f,)

def get_5class_per_class_vector(shap_arr, sample_i, c):
    '''SHAP values for class c.'''
    return shap_arr[sample_i, :, c]

def compute_pair_metrics_5class(arr_a, arr_b, k, sample_indices, mode, class_c=None):
    '''mode: "aggregate" or "perclass". For perclass, pass class_c.'''
    if len(sample_indices) == 0:
        return {m: np.nan for m in METRICS}
    out = {m: 0.0 for m in METRICS}
    for i in sample_indices:
        if mode == 'aggregate':
            sv_a = get_5class_aggregate_vector(arr_a, i)
            sv_b = get_5class_aggregate_vector(arr_b, i)
        else:
            sv_a = get_5class_per_class_vector(arr_a, i, class_c)
            sv_b = get_5class_per_class_vector(arr_b, i, class_c)
        for m, fn in METRICS.items():
            out[m] += fn(sv_a, sv_b, k)
    return {m: out[m] / len(sample_indices) for m in METRICS}

print('Computing Krishna metrics — 5-class aggregate...\n')
results_5class_agg = []
all_idx = np.arange(SHAP_5CLASS['rf'].shape[0])
for (arch_a, arch_b) in PAIRS:
    arr_a = SHAP_5CLASS[arch_a]
    arr_b = SHAP_5CLASS[arch_b]
    for k in K_VALUES:
        m = compute_pair_metrics_5class(arr_a, arr_b, k, all_idx, 'aggregate')
        row = {'Pair': f'{arch_a}-{arch_b}', 'k': k, 'Class': 'aggregate', 'Target': '5class'}
        row.update(m)
        results_5class_agg.append(row)
        print(f'  {arch_a}-{arch_b} @ k={k}: ' + ', '.join(f'{mn[:8]}={v:.3f}' for mn, v in m.items()))

df_5class_agg = pd.DataFrame(results_5class_agg)

Computing Krishna metrics — 5-class aggregate...

  rf-xgb @ k=5: feature_=0.641, rank_agr=0.225, sign_agr=0.999, signed_r=0.225, rank_cor=0.559, pairwise=0.728
  rf-xgb @ k=10: feature_=0.747, rank_agr=0.168, sign_agr=1.000, signed_r=0.168, rank_cor=0.551, pairwise=0.714


In [ ]:
print('Computing Krishna metrics — 5-class per-class...\n')
results_5class_pc = []
for c in range(5):
    class_idx = np.where(y_sub_5 == c)[0]
    if len(class_idx) < 5:
        print(f'  Skipping {CLASS_NAMES_5[c]}: only {len(class_idx)} samples')
        continue
    for (arch_a, arch_b) in PAIRS:
        arr_a = SHAP_5CLASS[arch_a]
        arr_b = SHAP_5CLASS[arch_b]
        for k in K_VALUES:
            m = compute_pair_metrics_5class(arr_a, arr_b, k, class_idx, 'perclass', class_c=c)
            row = {'Pair': f'{arch_a}-{arch_b}', 'k': k, 'Class': CLASS_NAMES_5[c], 'Target': '5class'}
            row.update(m)
            results_5class_pc.append(row)
    print(f'  ✓ {CLASS_NAMES_5[c]} (n={len(class_idx)})')

df_5class_pc = pd.DataFrame(results_5class_pc)

---
## Step 5 — Combine and save

In [ ]:
# Full combined table
all_results = pd.concat([df_binary, df_5class_agg, df_5class_pc], ignore_index=True)
all_results.to_csv(TABLES_DIR / 'nslkdd_krishna_full.csv', index=False)

# Aggregate-only table
agg = pd.concat([df_binary, df_5class_agg], ignore_index=True)
agg.to_csv(TABLES_DIR / 'nslkdd_krishna_aggregate.csv', index=False)

# Per-class table
df_5class_pc.to_csv(TABLES_DIR / 'nslkdd_krishna_perclass.csv', index=False)

# Headline summary (k=10 only, aggregate level)
headline = agg[agg['k'] == 10][['Target', 'Pair'] + list(METRICS.keys())]
headline.to_csv(TABLES_DIR / 'nslkdd_krishna_summary.csv', index=False)

print('=== AGGREGATE RESULTS (k=10) ===\n')
print(headline.to_string(index=False, float_format='%.3f'))
print(f'\n✓ Saved 4 tables to {TABLES_DIR}')

In [ ]:
# Per-class summary at k=10
print('\n=== PER-CLASS 5-CLASS RESULTS (k=10) ===\n')
pc10 = df_5class_pc[df_5class_pc['k'] == 10][['Class', 'Pair'] + list(METRICS.keys())]
print(pc10.to_string(index=False, float_format='%.3f'))

---
## Step 6 — k-robustness check

Do the agreement metrics depend heavily on k? If results are similar across k=5, 10, 15, the claims are robust.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, ['binary', '5class']):
    sub = agg[agg['Target'] == target].copy()
    sub['Metric'] = 'feature_agreement'  # we'll plot all metrics but loop

    for metric in ['feature_agreement', 'rank_correlation', 'pairwise_rank_agreement']:
        for pair in sub['Pair'].unique():
            data = sub[sub['Pair'] == pair].sort_values('k')
            ax.plot(data['k'], data[metric], marker='o', label=f'{pair} | {metric}')
    ax.set_xlabel('k')
    ax.set_ylabel('Metric value')
    ax.set_title(f'Cross-model agreement vs k ({target})')
    ax.set_xticks(K_VALUES)
    ax.legend(fontsize=7, loc='lower right', ncol=2)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIG_DIR / 'nslkdd_krishna_k_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 7 — Headline heatmaps

Two heatmaps (one per target). Rows are model pairs, columns are metrics. k=10.

In [ ]:
for target in ['binary', '5class']:
    df = agg[(agg['Target'] == target) & (agg['k'] == 10)].copy()
    metric_cols = list(METRICS.keys())
    pivot = df.set_index('Pair')[metric_cols]

    fig, ax = plt.subplots(figsize=(11, 4))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
                cbar_kws={'label': 'Agreement'}, ax=ax)
    ax.set_title(f'Krishna agreement metrics — {target} (k=10)')
    ax.set_xlabel('Metric')
    ax.set_ylabel('Model pair')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'nslkdd_krishna_heatmap_{target}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Step 8 — Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/06_shap_agreement.ipynb
!git add results/
!git status --short
!git commit -m "Notebook 06: cross-model SHAP agreement (Krishna metrics, k=5/10/15, per-class)"
!git push

---
## What to look for

**Headline metrics (aggregate, k=10):**
- Feature Agreement around 0.4-0.7 is typical for cross-model XAI comparisons
- Rank Agreement is usually much lower (0.05-0.20) — same features but different order is common
- Rank Correlation around 0.3-0.7 indicates moderate-to-strong overall ranking similarity
- Pairwise Rank Agreement around 0.7-0.9 — the most robust metric, since it tests pairwise ordering

**Per-class patterns:**
- The rare classes (R2L, U2R) usually show LOWER agreement than common classes — models learn rare patterns differently
- DoS typically shows HIGHEST agreement — easy class, all models converge

**k-robustness:**
- All curves should be relatively flat or monotonic across k=5/10/15
- Wild swings would indicate the agreement is artefactual

## Next notebook (07)

**SCTS-v2** — combine calibration confidence + stability score (from Notebook 05) + conformal coverage into a single per-alert trust score from 0-100.
